### Street View Blurring System YOLOv8


In [1]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
!{sys.executable} -m pip install --upgrade --force-reinstall nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached traitlets-5.14.3-py3-none-any.whl.metadata (10 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached traitlets-5.14.3-py3-none-any.whl (85 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: fastjsonschema
    Found existing installation: fastjsonschema 2.21.1
    Uninstalling fastjsonschema-2.21.1:
      Successfully uninstalled fastjsonschema-2.21.1
  Attempting uninstall: typing-

In [2]:
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
import ultralytics
import cv2
from ultralytics import YOLO

# Verify system setup for training
ultralytics.checks()

Ultralytics 8.4.11 🚀 Python-3.9.6 torch-2.8.0 CPU (Apple M5 Pro)
Setup complete ✅ (15 CPUs, 24.0 GB RAM, 95.9/926.3 GB disk)


In [4]:
from pathlib import Path

base_path = Path("./data")

def check_dataset_consistency(base):
    splits = ['train', 'val', 'test']

    print(f"{'Split':<10} | {'Images':<10} | {'Labels':<10} | {'Status'}")
    print("-" * 50)

    for split in splits:
        # Correct pathing based on your screenshot:
        # Data/images/train/ and Data/labels/train/
        img_folder = base / 'images' / split
        lbl_folder = base / 'labels' / split

        # Count files (handling case sensitivity and common extensions)
        img_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']
        img_count = sum(len(list(img_folder.glob(ext))) for ext in img_extensions)
        lbl_count = len(list(lbl_folder.glob('*.txt')))

        status = "✅ Match" if img_count == lbl_count and img_count > 0 else "❌ Mismatch or Empty"
        print(f"{split:<10} | {img_count:<10} | {lbl_count:<10} | {status}")

check_dataset_consistency(base_path)

Split      | Images     | Labels     | Status
--------------------------------------------------
train      | 23604      | 23604      | ✅ Match
val        | 1073       | 1073       | ✅ Match
test       | 386        | 386        | ✅ Match


In [5]:
#converting YOLO coordinates to pixel coordinates

import cv2
import matplotlib.pyplot as plt

def audit_sample(image_path, label_path):
    # Load image
    img = cv2.imread(str(image_path))
    h, w, _ = img.shape

    # Read YOLO label (class, cx, cy, bw, bh)
    with open(label_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        data = line.split()
        cx, cy, bw, bh = map(float, data[1:])

        # BUSINESS LOGIC: Convert normalized to pixel coordinates
        x1 = int((cx - bw/2) * w)
        y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w)
        y2 = int((cy + bh/2) * h)

        # Draw green bounding box
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, "License Plate", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Data Audit: Bounding Box Verification")
    plt.axis('off')
    plt.show()

# Test on one image from your 25,470 training set
# Update filenames to real ones in your folder
# audit_sample(data_splits['train']/'images'/'example.jpg', data_splits['train']/'labels'/'example.txt')

## Creating config yaml

In [7]:
import yaml

# Use .resolve() to get the full absolute path on your Mac
abs_base = base_path.resolve()

yaml_content = {
    'train': str(abs_base / 'images' / 'train'),
    'val': str(abs_base / 'images' / 'val'),
    'test': str(abs_base / 'images' / 'test'),
    'nc': 1,
    'names': ['license_plate']
}

with open('license_plate.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

print("✅ license_plate.yaml created successfully.")

✅ license_plate.yaml created successfully.


## Training phase

In [8]:
 from ultralytics import YOLO

# 1. Load the Model
# We use 'yolov8n.pt' (Nano) because it balances speed and accuracy
# and is less likely to overfit on our reduced dataset of ~2,100 images.
model = YOLO('yolov8n.pt')

# 2. Train the Model
results = model.train(
    data='license_plate.yaml',    # The config file we created earlier
    epochs=50,                   # 50 iterations to allow the model to learn features
    imgsz=640,                   # Resize images to 640x640 for consistency
    batch=16,                    # Process 16 images at a time
    device='mps',                # Change to 'cpu' if not on Apple Silicon
    name='plate_anonymizer_v1'   # Folder name for your logs and weights
)

/Users/kishankunal/workspace/AppliedAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.11 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M5 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=license_plate.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_anonymizer_v13, nbs=64, nms=False, op